In [1]:
CHECKPOINT_PATH = "examples/imagenet/ctm_imagenet_D=4096_T=50_M=25.pt"

In [2]:
import math
import argparse
import datetime
import numpy as np
import os
import time
from pathlib import Path
import torch.nn.functional as F
import torch
import torch.backends.cudnn as cudnn
# from torch.utils.tensorboard import SummaryWriter
import torchvision.transforms as transforms
import torchvision.datasets as datasets

# from jit_packages.util.crop import center_crop_arr
# import jit_packages.util.misc as misc

import copy
# from jit_packages.engine_jit import train_one_epoch, evaluate

from jit_packages.denoiser import Denoiser

import cv2
import os
import matplotlib.pyplot as plt
from PIL import Image
from tqdm import tqdm

import zipfile
import json
import random
import numpy as np
from PIL import Image
import torch
import torch.nn.functional as F
import lpips

import os
import math
import numpy as np
import cv2
import imageio.v2 as imageio

In [3]:
import torch
import matplotlib.pyplot as plt
from PIL import Image
from torchvision import transforms
from models.ctm import ContinuousThoughtMachine as CTM
import urllib
from IPython.display import Image as IPyImage, display
from pprint import pprint
import numpy as np
import seaborn as sns
import os
import torch.nn.functional as F
from matplotlib import patheffects
from scipy import ndimage
import imageio
import mediapy
from tqdm import tqdm
from tasks.image_classification.plotting import plot_neural_dynamics
from tasks.image_classification.imagenet_classes import IMAGENET2012_CLASSES

In [4]:
# set seed
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)  
    torch.manual_seed(seed)  
    torch.cuda.manual_seed(seed)  
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


### Load the Model

In [5]:
device = torch.device("cuda:1" if torch.cuda.is_available() else "cpu")
checkpoint = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)

# The args used for instantiating the model
model_args = checkpoint['args']

We can see what hyperparameters were used with this CTM

In [6]:
pprint(vars(model_args))

{'backbone_type': 'resnet152-4',
 'd_input': 1024,
 'd_model': 4096,
 'deep_memory': True,
 'do_normalisation': False,
 'heads': 16,
 'iterations': 50,
 'memory_hidden_dims': 64,
 'memory_length': 25,
 'n_random_pairing_self': 32,
 'n_synch_action': 2048,
 'n_synch_out': 8196,
 'neuron_select_type': 'random-pairing',
 'out_dims': 1000,
 'positional_embedding_type': 'none',
 'synapse_depth': 8}


This tell us that the CTM has:
- 4096 neurons
- 50 internal ticks
- 25 memory length
- 8196 pairs of neurons are used for synchronization
- Uses a ResNet backbone (preprocessor)

In [7]:
model_ctm = CTM(
    iterations=model_args.iterations,
    d_model=model_args.d_model,
    d_input=model_args.d_input,
    heads=model_args.heads,
    n_synch_out=model_args.n_synch_out,
    n_synch_action=model_args.n_synch_action,
    synapse_depth=model_args.synapse_depth,
    memory_length=model_args.memory_length,
    deep_nlms=model_args.deep_memory,
    memory_hidden_dims=model_args.memory_hidden_dims,
    do_layernorm_nlm=model_args.do_normalisation,
    backbone_type=model_args.backbone_type,
    positional_embedding_type=model_args.positional_embedding_type,
    out_dims=model_args.out_dims,
    prediction_reshaper=[-1],
    dropout=0,
    neuron_select_type=model_args.neuron_select_type,
    n_random_pairing_self=model_args.n_random_pairing_self,
).to(device)

load_result = model_ctm.load_state_dict(checkpoint['model_state_dict'], strict=False)
model_ctm.eval(); # Set model to evaluation mode

/home/yang/anaconda3/envs/conda_dl/lib/python3.8/site-packages/torch/nn/modules/lazy.py:181: UserWarning: Lazy modules are a new feature under heavy development so changes to the API or functionality can happen at any moment.
  warnings.warn('Lazy modules are a new feature under heavy development '


Using neuron select type: random-pairing
Synch representation size action: 2048
Synch representation size out: 8196


In [8]:
def count_parameters(model):
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    print(f"Total parameters: {total_params:,}")
    print(f"Trainable parameters: {trainable_params:,}")
    print(f"Non-trainable parameters: {total_params - trainable_params:,}")



In [9]:
def get_args_parser():
    parser = argparse.ArgumentParser('JiT', add_help=False)

    # architecture
    parser.add_argument('--model', default='JiT-H/16', type=str, metavar='MODEL',
                        help='Name of the model to train')
    parser.add_argument('--img_size', default=256, type=int, help='Image size')
    parser.add_argument('--attn_dropout', type=float, default=0.0, help='Attention dropout rate')
    parser.add_argument('--proj_dropout', type=float, default=0.0, help='Projection dropout rate')

    # training
    parser.add_argument('--epochs', default=200, type=int)
    parser.add_argument('--warmup_epochs', type=int, default=5, metavar='N',
                        help='Epochs to warm up LR')
    parser.add_argument('--batch_size', default=128, type=int,
                        help='Batch size per GPU (effective batch size = batch_size * # GPUs)')
    parser.add_argument('--lr', type=float, default=None, metavar='LR',
                        help='Learning rate (absolute)')
    parser.add_argument('--blr', type=float, default=5e-5, metavar='LR',
                        help='Base learning rate: absolute_lr = base_lr * total_batch_size / 256')
    parser.add_argument('--min_lr', type=float, default=0., metavar='LR',
                        help='Minimum LR for cyclic schedulers that hit 0')
    parser.add_argument('--lr_schedule', type=str, default='constant',
                        help='Learning rate schedule')
    parser.add_argument('--weight_decay', type=float, default=0.0,
                        help='Weight decay (default: 0.0)')
    parser.add_argument('--ema_decay1', type=float, default=0.9999,
                        help='The first ema to track. Use the first ema for sampling by default.')
    parser.add_argument('--ema_decay2', type=float, default=0.9996,
                        help='The second ema to track')
    parser.add_argument('--P_mean', default=-0.8, type=float)
    parser.add_argument('--P_std', default=0.8, type=float)
    parser.add_argument('--noise_scale', default=1.0, type=float)
    parser.add_argument('--t_eps', default=5e-2, type=float)
    parser.add_argument('--label_drop_prob', default=0.1, type=float)

    parser.add_argument('--seed', default=0, type=int)
    parser.add_argument('--start_epoch', default=0, type=int, metavar='N',
                        help='Starting epoch')
    parser.add_argument('--num_workers', default=12, type=int)
    parser.add_argument('--pin_mem', action='store_true',
                        help='Pin CPU memory in DataLoader for faster GPU transfers')
    parser.add_argument('--no_pin_mem', action='store_false', dest='pin_mem')
    parser.set_defaults(pin_mem=True)

    # sampling
    parser.add_argument('--sampling_method', default='heun', type=str,
                        help='ODE samping method')
    parser.add_argument('--num_sampling_steps', default=50, type=int,
                        help='Sampling steps')
    parser.add_argument('--cfg', default=3.0, type=float,
                        help='Classifier-free guidance factor')
    parser.add_argument('--interval_min', default=0.1, type=float,
                        help='CFG interval min')
    parser.add_argument('--interval_max', default=1.0, type=float,
                        help='CFG interval max')
    parser.add_argument('--num_images', default=50000, type=int,
                        help='Number of images to generate')
    parser.add_argument('--eval_freq', type=int, default=40,
                        help='Frequency (in epochs) for evaluation')
    parser.add_argument('--online_eval', action='store_true')
    parser.add_argument('--evaluate_gen', action='store_true')
    parser.add_argument('--gen_bsz', type=int, default=256,
                        help='Generation batch size')

    # dataset
    parser.add_argument('--data_path', default='imagenet', type=str,
                        help='Path to the dataset')
    parser.add_argument('--class_num', default=1000, type=int)

    # checkpointing
    parser.add_argument('--output_dir', default='./output_dir',
                        help='Directory to save outputs (empty for no saving)')
    parser.add_argument('--resume', default='',
                        help='Folder that contains checkpoint to resume from')
    parser.add_argument('--save_last_freq', type=int, default=5,
                        help='Frequency (in epochs) to save checkpoints')
    parser.add_argument('--log_freq', default=100, type=int)
    parser.add_argument('--device', default='cuda',
                        help='Device to use for training/testing')

    # distributed training
    parser.add_argument('--world_size', default=1, type=int,
                        help='Number of distributed processes')
    parser.add_argument('--local_rank', default=-1, type=int)
    parser.add_argument('--dist_on_itp', action='store_true')
    parser.add_argument('--dist_url', default='env://',
                        help='URL used to set up distributed training')

    return parser

In [10]:
def save_fig_jit(sampled_images, path = 'generated_figs'):
    sampled_images = (sampled_images + 1) / 2
    sampled_images = sampled_images.detach().cpu()
    for b_id in range(sampled_images.size(0)):
        gen_img = np.round(np.clip(sampled_images[b_id].numpy().transpose([1, 2, 0]) * 255, 0, 255))
        gen_img = gen_img.astype(np.uint8)[:, :, ::-1]
        cv2.imwrite(f'{path}/jit_{b_id}.png', gen_img)

In [11]:
def save_traj_grid_compare(
    traj_uncond,
    traj_cond,
    sampled_images,
    save_dir,
    prefix="traj",
    nrow=None,
    gap=20,
    title_h=40,
    gif_name=None,
    gif_duration=0.15,
):
    """
    Layout:
        top-left     : uncond tweedie
        top-right    : cond tweedie
        bottom-left  : generated images (static across gif)
        bottom-right : blank

    Args:
        traj_uncond: list of tensors [B,3,H,W], assumed in [-1,1]
        traj_cond:   list of tensors [B,3,H,W], assumed in [-1,1]
        sampled_images: tensor [B,3,H,W], assumed in [-1,1]
        save_dir: output folder
        prefix: filename prefix
        nrow: number of images per row in each panel
        gap: blank space between panels
        title_h: title bar height for each panel
        gif_name: output gif filename; default f"{prefix}.gif"
        gif_duration: seconds per frame
    """
    os.makedirs(save_dir, exist_ok=True)

    assert len(traj_uncond) == len(traj_cond)
    assert len(traj_uncond) > 0

    steps = len(traj_uncond)
    B, C, H, W = traj_uncond[0].shape

    if nrow is None:
        nrow = int(math.sqrt(B))
        if nrow * nrow < B:
            nrow += 1
    ncol = math.ceil(B / nrow)

    grid_h = ncol * H
    grid_w = nrow * W
    panel_h = title_h + grid_h
    panel_w = grid_w

    canvas_h = panel_h * 2 + gap
    canvas_w = panel_w * 2 + gap

    def make_grid(imgs):
        """
        imgs: tensor [B,3,H,W], assumed in [-1,1]
        return: uint8 RGB image [grid_h, grid_w, 3]
        """
        imgs = (imgs + 1) / 2
        imgs = imgs.clamp(0, 1)
        imgs = imgs.detach().cpu()

        grid = np.zeros((grid_h, grid_w, 3), dtype=np.uint8)

        for i in range(B):
            r = i // nrow
            c = i % nrow

            img = imgs[i].numpy().transpose(1, 2, 0)
            img = (img * 255).round().astype(np.uint8)

            grid[
                r * H:(r + 1) * H,
                c * W:(c + 1) * W
            ] = img

        return grid

    def draw_title(canvas_bgr, text, x0, y0):
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.8
        thickness = 2

        (tw, th), _ = cv2.getTextSize(text, font, font_scale, thickness)
        x = x0 + max((panel_w - tw) // 2, 5)
        y = y0 + max((title_h + th) // 2, th + 5)

        cv2.putText(
            canvas_bgr,
            text,
            (x, y),
            font,
            font_scale,
            (255, 255, 255),
            thickness,
            cv2.LINE_AA,
        )

    # static panel: generated images
    grid_generated = make_grid(sampled_images)

    saved_paths = []

    for step in range(steps):
        grid_uncond = make_grid(traj_uncond[step])
        grid_cond = make_grid(traj_cond[step])

        # RGB canvas
        canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)

        # top-left: uncond
        y0, x0 = 0, 0
        canvas[y0 + title_h:y0 + title_h + grid_h, x0:x0 + grid_w] = grid_uncond

        # top-right: cond
        y0, x0 = 0, panel_w + gap
        canvas[y0 + title_h:y0 + title_h + grid_h, x0:x0 + grid_w] = grid_cond

        # bottom-left: generated images (static)
        y0, x0 = panel_h + gap, 0
        canvas[y0 + title_h:y0 + title_h + grid_h, x0:x0 + grid_w] = grid_generated

        # bottom-right left blank

        # convert to BGR for cv2 text/write
        canvas_bgr = canvas[:, :, ::-1].copy()

        draw_title(canvas_bgr, f"uncond tweedie  |  timestep {step:03d}", 0, 0)
        draw_title(canvas_bgr, f"cond tweedie  |  timestep {step:03d}", panel_w + gap, 0)
        draw_title(canvas_bgr, "generated images", 0, panel_h + gap)

        filename = f"{prefix}_step{step:03d}.png"
        path = os.path.join(save_dir, filename)
        cv2.imwrite(path, canvas_bgr)
        saved_paths.append(path)

    if gif_name is None:
        gif_name = f"{prefix}.gif"

    gif_path = os.path.join(save_dir, gif_name)

    frames = []
    for p in saved_paths:
        frame_bgr = cv2.imread(p, cv2.IMREAD_COLOR)
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        frames.append(frame_rgb)

    # loop=0 means infinite loop
    imageio.mimsave(gif_path, frames, duration=gif_duration, )

    return gif_path

In [12]:
parser = get_args_parser()
args = parser.parse_args([])   # ← 在 notebook 里必须传一个空 list

In [13]:
model_jit = Denoiser(args)

In [14]:

model_jit.load_state_dict(torch.load('/home/yang/diffusion_model_with_double_guidance/fig_task/jit_dmdg/JiT/ckpts/checkpoint_jit_h_16.pth')['model'])
model_jit = model_jit.to(device)

In [15]:
count_parameters(model_ctm), count_parameters(model_jit)

Total parameters: 185,989,886
Trainable parameters: 185,989,886
Non-trainable parameters: 0
Total parameters: 952,842,048
Trainable parameters: 952,514,368
Non-trainable parameters: 327,680


(None, None)

In [16]:
# def sample_jit monitored by CTM
DATASET_MEAN = [0.485, 0.456, 0.406]
DATASET_STD = [0.229, 0.224, 0.225]


def jit_to_ctm(x_jit):
    """
    x_jit: [B,3,H,W], in [-1,1]
    return: [B,3,H,W], CTM-normalized
    """
    mean = torch.tensor(
        DATASET_MEAN, device=x_jit.device, dtype=x_jit.dtype
    ).view(1, 3, 1, 1)
    std = torch.tensor(
        DATASET_STD, device=x_jit.device, dtype=x_jit.dtype
    ).view(1, 3, 1, 1)

    x_01 = ((x_jit + 1.0) / 2.0).clamp(0.0, 1.0)
    x_ctm = (x_01 - mean) / std
    return x_ctm


@torch.no_grad()
def sample_jit(
    model_jit,
    labels,
    sample_steps=None,
    cfg_scale=None,
    method=None,
    return_traj=True,
    model_ctm=None,
    ctm_input_size=224,
    attn_output_size=256,
):
    device = labels.device
    bsz = labels.size(0)
    model_jit.eval()

    if model_ctm is not None:
        model_ctm.eval()
        ctm_device = next(model_ctm.parameters()).device
    else:
        ctm_device = None

    steps = model_jit.steps if sample_steps is None else sample_steps
    method = model_jit.method if method is None else method
    cfg_scale = model_jit.cfg_scale if cfg_scale is None else cfg_scale

    z = model_jit.noise_scale * torch.randn(
        bsz, 3, model_jit.img_size, model_jit.img_size, device=device
    )

    timesteps = torch.linspace(
        0.0, 1.0, steps + 1, device=device
    ).view(-1, *([1] * z.ndim)).expand(-1, bsz, -1, -1, -1)

    traj_x0_uncond = []
    traj_x0_cond = []

    # CTM outputs over diffusion timesteps
    ctm_predictions_uncond = []
    ctm_certainties_uncond = []
    ctm_attention_256_uncond = []

    ctm_predictions_cond = []
    ctm_certainties_cond = []
    ctm_attention_256_cond = []

    def run_ctm_on_jit_batch(x_jit_batch):
        """
        x_jit_batch: [B,3,256,256] in [-1,1]
        returns:
            predictions: [B,C,T_ctm]
            certainties: [B,2,T_ctm]
            attention_256: [T_ctm,B,attn_output_size,attn_output_size]
        """
        if model_ctm is None:
            return None, None, None

        x_ctm = jit_to_ctm(x_jit_batch)

        # 这里按你的要求直接用 bilinear: 256 -> 224
        x_ctm = F.interpolate(
            x_ctm,
            size=(ctm_input_size, ctm_input_size),
            mode="bilinear",
            align_corners=False,
        ).to(ctm_device)

        predictions, certainties, synchronization, pre_activations, post_activations, attention_tracking = model_ctm(
            x_ctm, track=True
        )

        # attention_tracking: [T_ctm,B,num_heads,1,196]
        if isinstance(attention_tracking, np.ndarray):
            attention_tracking = torch.from_numpy(attention_tracking).to(ctm_device)

        attention_tracking = attention_tracking.squeeze(-2)   # [T_ctm,B,num_heads,196]
        attention_mean = attention_tracking.mean(dim=2)       # [T_ctm,B,196]

        h_feat, w_feat = model_ctm.kv_features.shape[-2:]
        T_ctm, B_ctm = attention_mean.shape[:2]

        attention_patch = attention_mean.reshape(T_ctm, B_ctm, h_feat, w_feat)

        attention_256 = F.interpolate(
            attention_patch.reshape(T_ctm * B_ctm, 1, h_feat, w_feat),
            size=(attn_output_size, attn_output_size),
            mode="bilinear",
            align_corners=False,
        ).reshape(T_ctm, B_ctm, attn_output_size, attn_output_size)

        return (
            predictions.detach().cpu(),
            certainties.detach().cpu(),
            attention_256.detach().cpu(),
        )

    def forward_sample_with_x0(z, t, labels):
        # unconditional branch
        null_labels = torch.full_like(labels, model_jit.num_classes)
        x_uncond = model_jit.net(z, t.flatten(), null_labels)
        v_uncond = (x_uncond - z) / (1.0 - t).clamp_min(model_jit.t_eps)

        # if cfg_scale == 0, do unconditional sampling only
        if cfg_scale == 0:
            x_cond = x_uncond * 0
            v_pred = v_uncond
            return v_pred, x_uncond, x_cond

        # conditional branch
        x_cond = model_jit.net(z, t.flatten(), labels)
        v_cond = (x_cond - z) / (1.0 - t).clamp_min(model_jit.t_eps)

        # cfg interval
        low, high = model_jit.cfg_interval
        interval_mask = (t < high) & ((low == 0) | (t > low))
        cfg_scale_interval = torch.where(
            interval_mask,
            torch.as_tensor(cfg_scale, device=device, dtype=z.dtype),
            torch.as_tensor(1.0, device=device, dtype=z.dtype),
        )

        v_pred = v_uncond + cfg_scale_interval * (v_cond - v_uncond)
        return v_pred, x_uncond, x_cond

    def euler_step_with_x0(z, t, t_next, labels):
        v_pred, x_uncond, x_cond = forward_sample_with_x0(z, t, labels)
        z_next = z + (t_next - t) * v_pred
        return z_next, x_uncond, x_cond

    def heun_step_with_x0(z, t, t_next, labels):
        v_pred_t, x_uncond_t, x_cond_t = forward_sample_with_x0(z, t, labels)

        z_next_euler = z + (t_next - t) * v_pred_t
        v_pred_t_next, _, _ = forward_sample_with_x0(z_next_euler, t_next, labels)

        v_pred = 0.5 * (v_pred_t + v_pred_t_next)
        z_next = z + (t_next - t) * v_pred
        return z_next, x_uncond_t, x_cond_t

    if method not in ["euler", "heun"]:
        raise NotImplementedError(f"Unknown sampling method: {method}")

    for i in tqdm(range(steps - 1)):
        t = timesteps[i]
        t_next = timesteps[i + 1]

        if method == "euler":
            z, x0_uncond, x0_cond = euler_step_with_x0(z, t, t_next, labels)
        else:
            z, x0_uncond, x0_cond = heun_step_with_x0(z, t, t_next, labels)

        if return_traj:
            traj_x0_uncond.append(x0_uncond.detach().clone())
            traj_x0_cond.append(x0_cond.detach().clone())

        if model_ctm is not None:
            pred_u, cert_u, attn_u = run_ctm_on_jit_batch(x0_uncond)
            pred_c, cert_c, attn_c = run_ctm_on_jit_batch(x0_cond)

            ctm_predictions_uncond.append(pred_u)
            ctm_certainties_uncond.append(cert_u)
            ctm_attention_256_uncond.append(attn_u)

            ctm_predictions_cond.append(pred_c)
            ctm_certainties_cond.append(cert_c)
            ctm_attention_256_cond.append(attn_c)

    # last step always euler
    z, x0_uncond, x0_cond = euler_step_with_x0(
        z, timesteps[-2], timesteps[-1], labels
    )

    if return_traj:
        traj_x0_uncond.append(x0_uncond.detach().clone())
        traj_x0_cond.append(x0_cond.detach().clone())
    else:
        traj_x0_uncond = None
        traj_x0_cond = None

    if model_ctm is not None:
        pred_u, cert_u, attn_u = run_ctm_on_jit_batch(x0_uncond)
        pred_c, cert_c, attn_c = run_ctm_on_jit_batch(x0_cond)

        ctm_predictions_uncond.append(pred_u)
        ctm_certainties_uncond.append(cert_u)
        ctm_attention_256_uncond.append(attn_u)

        ctm_predictions_cond.append(pred_c)
        ctm_certainties_cond.append(cert_c)
        ctm_attention_256_cond.append(attn_c)
    else:
        ctm_predictions_uncond = None
        ctm_certainties_uncond = None
        ctm_attention_256_uncond = None

        ctm_predictions_cond = None
        ctm_certainties_cond = None
        ctm_attention_256_cond = None

    ctm_outputs = {
        "uncond": {
            "predictions": ctm_predictions_uncond,
            "certainties": ctm_certainties_uncond,
            "attention_256": ctm_attention_256_uncond,
        },
        "cond": {
            "predictions": ctm_predictions_cond,
            "certainties": ctm_certainties_cond,
            "attention_256": ctm_attention_256_cond,
        },
    }

    return z, traj_x0_uncond, traj_x0_cond, ctm_outputs

In [26]:
with torch.inference_mode():
    set_seed(124)
    
    sampled_images, traj_x0_uncond, traj_x0_cond, ctm_outputs = sample_jit(
        model_jit = model_jit,
        labels= torch.tensor(torch.arange(0, 20, dtype=torch.int64, device=device), dtype=torch.int64, device=device),
        sample_steps = 50,
        cfg_scale = 3,
        method = "heun",
        return_traj = True,
        model_ctm = model_ctm,
        ctm_input_size = 224,
        attn_output_size = 256,
    )

/tmp/ipykernel_3862499/1212733629.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels= torch.tensor(torch.arange(0, 20, dtype=torch.int64, device=device), dtype=torch.int64, device=device),


In [27]:
def save_traj_grid_compare(
    traj_uncond,
    traj_cond,
    sampled_images,
    ctm_outputs,
    save_dir,
    prefix="traj",
    nrow=None,
    gap=20,
    title_h=40,
    gif_name=None,
    gif_duration=0.15,
    overlay_alpha=0.45,
):
    """
    Layout (3 columns x 2 rows):
        row 1:
            col 1: uncond tweedie
            col 2: cond tweedie
            col 3: generated images (static)
        row 2:
            col 1: uncond attention overlay
            col 2: cond attention overlay
            col 3: blank
    """
    os.makedirs(save_dir, exist_ok=True)

    assert len(traj_uncond) == len(traj_cond)
    assert len(traj_uncond) > 0
    assert len(ctm_outputs["uncond"]["attention_256"]) == len(traj_uncond)
    assert len(ctm_outputs["cond"]["attention_256"]) == len(traj_cond)

    steps = len(traj_uncond)
    B, C, H, W = traj_uncond[0].shape

    if nrow is None:
        nrow = int(math.sqrt(B))
        if nrow * nrow < B:
            nrow += 1
    ncol = math.ceil(B / nrow)

    grid_h = ncol * H
    grid_w = nrow * W

    panel_h = title_h + grid_h
    panel_w = grid_w

    canvas_h = panel_h * 2 + gap
    canvas_w = panel_w * 3 + gap * 2

    def to_rgb_uint8_batch(imgs):
        """
        imgs: tensor [B,3,H,W], in [-1,1]
        return: numpy [B,H,W,3], uint8 RGB
        """
        imgs = (imgs + 1) / 2
        imgs = imgs.clamp(0, 1)
        imgs = imgs.detach().cpu().numpy()
        imgs = np.transpose(imgs, (0, 2, 3, 1))
        imgs = np.round(imgs * 255).astype(np.uint8)
        return imgs

    def make_rgb_grid(imgs):
        """
        imgs: tensor [B,3,H,W], in [-1,1]
        return: uint8 RGB grid [grid_h, grid_w, 3]
        """
        imgs = to_rgb_uint8_batch(imgs)
        grid = np.zeros((grid_h, grid_w, 3), dtype=np.uint8)

        for i in range(B):
            r = i // nrow
            c = i % nrow
            grid[
                r * H:(r + 1) * H,
                c * W:(c + 1) * W
            ] = imgs[i]

        return grid

    def normalize_attn_map(attn):
        attn = attn.astype(np.float32)
        a_min = float(attn.min())
        a_max = float(attn.max())
        if a_max > a_min:
            attn = (attn - a_min) / (a_max - a_min)
        else:
            attn = np.zeros_like(attn, dtype=np.float32)
        return attn

    def make_attention_overlay_grid(imgs, attn_maps):
        """
        imgs: tensor [B,3,H,W], in [-1,1]
        attn_maps: tensor/np.array [B,H,W]
        return: uint8 RGB grid [grid_h, grid_w, 3]
        """
        imgs_rgb = to_rgb_uint8_batch(imgs)

        if isinstance(attn_maps, torch.Tensor):
            attn_maps = attn_maps.detach().cpu().numpy()

        grid = np.zeros((grid_h, grid_w, 3), dtype=np.uint8)

        for i in range(B):
            r = i // nrow
            c = i % nrow

            base_img = imgs_rgb[i]  # RGB uint8
            attn = normalize_attn_map(attn_maps[i])

            attn_uint8 = np.round(attn * 255).astype(np.uint8)
            heat_bgr = cv2.applyColorMap(attn_uint8, cv2.COLORMAP_JET)
            heat_rgb = cv2.cvtColor(heat_bgr, cv2.COLOR_BGR2RGB)

            overlay = cv2.addWeighted(
                base_img, 1.0 - overlay_alpha,
                heat_rgb, overlay_alpha,
                0
            )

            grid[
                r * H:(r + 1) * H,
                c * W:(c + 1) * W
            ] = overlay

        return grid

    def draw_title(canvas_bgr, text, x0, y0):
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.7
        thickness = 2

        (tw, th), _ = cv2.getTextSize(text, font, font_scale, thickness)
        x = x0 + max((panel_w - tw) // 2, 5)
        y = y0 + max((title_h + th) // 2, th + 5)

        cv2.putText(
            canvas_bgr,
            text,
            (x, y),
            font,
            font_scale,
            (255, 255, 255),
            thickness,
            cv2.LINE_AA,
        )

    grid_generated = make_rgb_grid(sampled_images)
    saved_paths = []

    for step in range(steps):
        grid_uncond = make_rgb_grid(traj_uncond[step])
        grid_cond = make_rgb_grid(traj_cond[step])

        # attention_256[step]: [T_ctm, B, 256, 256]
        attn_uncond_step = ctm_outputs["uncond"]["attention_256"][step].mean(dim=0)  # [B,256,256]
        attn_cond_step = ctm_outputs["cond"]["attention_256"][step].mean(dim=0)      # [B,256,256]

        grid_attn_uncond = make_attention_overlay_grid(traj_uncond[step], attn_uncond_step)
        grid_attn_cond = make_attention_overlay_grid(traj_cond[step], attn_cond_step)

        cert_uncond_step = ctm_outputs["uncond"]["certainties"][step]
        cert_cond_step = ctm_outputs["cond"]["certainties"][step]

        if isinstance(cert_uncond_step, np.ndarray):
            cert_uncond_step = torch.from_numpy(cert_uncond_step)
        if isinstance(cert_cond_step, np.ndarray):
            cert_cond_step = torch.from_numpy(cert_cond_step)

        cert_uncond_mean = float(cert_uncond_step[:, 1, :].mean().item())
        cert_cond_mean = float(cert_cond_step[:, 1, :].mean().item())

        canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)

        # row 1
        y0, x0 = 0, 0
        canvas[y0 + title_h:y0 + title_h + grid_h, x0:x0 + grid_w] = grid_uncond

        y0, x0 = 0, panel_w + gap
        canvas[y0 + title_h:y0 + title_h + grid_h, x0:x0 + grid_w] = grid_cond

        y0, x0 = 0, (panel_w + gap) * 2
        canvas[y0 + title_h:y0 + title_h + grid_h, x0:x0 + grid_w] = grid_generated

        # row 2
        y0, x0 = panel_h + gap, 0
        canvas[y0 + title_h:y0 + title_h + grid_h, x0:x0 + grid_w] = grid_attn_uncond

        y0, x0 = panel_h + gap, panel_w + gap
        canvas[y0 + title_h:y0 + title_h + grid_h, x0:x0 + grid_w] = grid_attn_cond

        canvas_bgr = canvas[:, :, ::-1].copy()

        draw_title(canvas_bgr, f"uncond tweedie | jit timestep {step:03d}", 0, 0)
        draw_title(canvas_bgr, f"cond tweedie | jit timestep {step:03d}", panel_w + gap, 0)
        draw_title(canvas_bgr, "generated images", (panel_w + gap) * 2, 0)

        draw_title(
            canvas_bgr,
            f"uncond attention overlay | jit timestep {step:03d} | cert2 avg {cert_uncond_mean:.4f}",
            0,
            panel_h + gap,
        )
        draw_title(
            canvas_bgr,
            f"cond attention overlay | jit timestep {step:03d} | cert2 avg {cert_cond_mean:.4f}",
            panel_w + gap,
            panel_h + gap,
        )

        filename = f"{prefix}_step{step:03d}.png"
        path = os.path.join(save_dir, filename)
        cv2.imwrite(path, canvas_bgr)
        saved_paths.append(path)

    if gif_name is None:
        gif_name = f"{prefix}.gif"

    gif_path = os.path.join(save_dir, gif_name)

    frames = []
    for p in saved_paths:
        frame_bgr = cv2.imread(p, cv2.IMREAD_COLOR)
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        frames.append(frame_rgb)


    imageio.mimsave(gif_path, frames, duration=gif_duration, loop=0)

    return gif_path

In [28]:
gif_path = save_traj_grid_compare(
    traj_uncond=traj_x0_uncond,
    traj_cond=traj_x0_cond,
    sampled_images=sampled_images,
    ctm_outputs=ctm_outputs,
    save_dir="tweedie_trajs",
    prefix="jit_compare",
    gif_duration=1.,
)
print(gif_path)

tweedie_trajs/jit_compare.gif


In [ ]:
# def sample_jit monitored by CTM
DATASET_MEAN = [0.485, 0.456, 0.406]
DATASET_STD = [0.229, 0.224, 0.225]


def jit_to_ctm(x_jit):
    """
    x_jit: [B,3,H,W], in [-1,1]
    return: [B,3,H,W], CTM-normalized
    """
    mean = torch.tensor(
        DATASET_MEAN, device=x_jit.device, dtype=x_jit.dtype
    ).view(1, 3, 1, 1)
    std = torch.tensor(
        DATASET_STD, device=x_jit.device, dtype=x_jit.dtype
    ).view(1, 3, 1, 1)

    x_01 = ((x_jit + 1.0) / 2.0).clamp(0.0, 1.0)
    x_ctm = (x_01 - mean) / std
    return x_ctm


@torch.no_grad()
def sample_ctm_jit(
    model_jit,
    labels,
    sample_steps=None,
    cfg_scale=None,
    method=None,
    return_traj=True,
    model_ctm=None,
    ctm_input_size=224,
    attn_output_size=256,
    attn_cert_tau=4.0,
    attn_mask_strength=0.05,
    attn_smooth_kernel=9,
    attn_mask_clip_min=0.95,
    attn_mask_clip_max=1.05,
):
    device = labels.device
    bsz = labels.size(0)
    model_jit.eval()

    if model_ctm is not None:
        model_ctm.eval()
        ctm_device = next(model_ctm.parameters()).device
    else:
        ctm_device = None

    steps = model_jit.steps if sample_steps is None else sample_steps
    method = model_jit.method if method is None else method
    cfg_scale = model_jit.cfg_scale if cfg_scale is None else cfg_scale

    z = model_jit.noise_scale * torch.randn(
        bsz, 3, model_jit.img_size, model_jit.img_size, device=device
    )

    timesteps = torch.linspace(
        0.0, 1.0, steps + 1, device=device
    ).view(-1, *([1] * z.ndim)).expand(-1, bsz, -1, -1, -1)

    traj_x0_uncond = []
    traj_x0_cond = []

    # CTM outputs over diffusion timesteps
    ctm_predictions_uncond = []
    ctm_certainties_uncond = []
    ctm_attention_256_uncond = []

    ctm_predictions_cond = []
    ctm_certainties_cond = []
    ctm_attention_256_cond = []
    ctm_spatial_mask_cond = []

    def run_ctm_on_jit_batch(x_jit_batch):
        """
        x_jit_batch: [B,3,256,256] in [-1,1]
        returns:
            predictions: [B,C,T_ctm]
            certainties: [B,2,T_ctm]
            attention_256: [T_ctm,B,attn_output_size,attn_output_size]
        """
        if model_ctm is None:
            return None, None, None

        x_ctm = jit_to_ctm(x_jit_batch)

        # 这里按你的要求直接用 bilinear: 256 -> 224
        x_ctm = F.interpolate(
            x_ctm,
            size=(ctm_input_size, ctm_input_size),
            mode="bilinear",
            align_corners=False,
        ).to(ctm_device)

        predictions, certainties, synchronization, pre_activations, post_activations, attention_tracking = model_ctm(
            x_ctm, track=True
        )

        # attention_tracking: [T_ctm,B,num_heads,1,196]
        if isinstance(attention_tracking, np.ndarray):
            attention_tracking = torch.from_numpy(attention_tracking).to(ctm_device)

        attention_tracking = attention_tracking.squeeze(-2)   # [T_ctm,B,num_heads,196]
        attention_mean = attention_tracking.mean(dim=2)       # [T_ctm,B,196]

        h_feat, w_feat = model_ctm.kv_features.shape[-2:]
        T_ctm, B_ctm = attention_mean.shape[:2]

        attention_patch = attention_mean.reshape(T_ctm, B_ctm, h_feat, w_feat)

        attention_256 = F.interpolate(
            attention_patch.reshape(T_ctm * B_ctm, 1, h_feat, w_feat),
            size=(attn_output_size, attn_output_size),
            mode="bilinear",
            align_corners=False,
        ).reshape(T_ctm, B_ctm, attn_output_size, attn_output_size)

        return (
            predictions.detach().cpu(),
            certainties.detach().cpu(),
            attention_256.detach().cpu(),
        )

    def build_spatial_cfg_mask_from_cond_ctm(cert_cond, attn_cond):
        """
        cert_cond: [B,2,T_ctm] on cpu
        attn_cond: [T_ctm,B,H,W] on cpu

        return:
            spatial_mask: [B,1,img_size,img_size] on device
        """
        if cert_cond is None or attn_cond is None:
            return torch.ones(
                (bsz, 1, model_jit.img_size, model_jit.img_size),
                device=device,
                dtype=z.dtype,
            )

        # certainty channel 1 as temporal weights
        cert2 = cert_cond[:, 1, :].float()  # [B, T_ctm]

        # softmax over CTM time dimension
        attn_weights = torch.softmax(attn_cert_tau * cert2, dim=-1)  # [B, T_ctm]

        # attn_cond: [T_ctm, B, H, W] -> [B, T_ctm, H, W]
        attn_bt = attn_cond.permute(1, 0, 2, 3).float()  # [B, T_ctm, H, W]

        # certainty-weighted aggregation over CTM time
        attn_agg = (attn_bt * attn_weights[:, :, None, None]).sum(dim=1)  # [B, H, W]

        # move to target device/dtype
        attn_agg = attn_agg.to(device=device, dtype=z.dtype).unsqueeze(1)  # [B,1,H,W]

        # optional smoothing
        if attn_smooth_kernel is not None and attn_smooth_kernel > 1:
            k = int(attn_smooth_kernel)
            if k % 2 == 0:
                k += 1
            attn_agg = F.avg_pool2d(
                attn_agg,
                kernel_size=k,
                stride=1,
                padding=k // 2,
            )

        # resize if needed
        if attn_agg.shape[-2:] != (model_jit.img_size, model_jit.img_size):
            attn_agg = F.interpolate(
                attn_agg,
                size=(model_jit.img_size, model_jit.img_size),
                mode="bilinear",
                align_corners=False,
            )

        # normalize to mean=1
        spatial_mask = attn_agg / (attn_agg.mean(dim=(-2, -1), keepdim=True) + 1e-6)

        # shrink toward 1.0 to keep it conservative
        spatial_mask = 1.0 + attn_mask_strength * (spatial_mask - 1.0)

        # clamp to safe range
        spatial_mask = spatial_mask.clamp(attn_mask_clip_min, attn_mask_clip_max)

        return spatial_mask

    def forward_sample_with_x0(z, t, labels):
        # unconditional branch
        null_labels = torch.full_like(labels, model_jit.num_classes)
        x_uncond = model_jit.net(z, t.flatten(), null_labels)
        v_uncond = (x_uncond - z) / (1.0 - t).clamp_min(model_jit.t_eps)

        # if cfg_scale == 0, do unconditional sampling only
        if cfg_scale == 0:
            x_cond = x_uncond * 0
            v_pred = v_uncond
            spatial_mask = torch.ones(
                (bsz, 1, model_jit.img_size, model_jit.img_size),
                device=device,
                dtype=z.dtype,
            )
            return v_pred, x_uncond, x_cond, spatial_mask

        # conditional branch
        x_cond = model_jit.net(z, t.flatten(), labels)
        v_cond = (x_cond - z) / (1.0 - t).clamp_min(model_jit.t_eps)

        # default spatial mask = all ones
        spatial_mask = torch.ones(
            (bsz, 1, model_jit.img_size, model_jit.img_size),
            device=device,
            dtype=z.dtype,
        )

        # only build CTM-based spatial mask if CTM exists
        if model_ctm is not None:
            _, cert_c, attn_c = run_ctm_on_jit_batch(x_cond)
            spatial_mask = build_spatial_cfg_mask_from_cond_ctm(cert_c, attn_c)
            # print(spatial_mask.mean(), spatial_mask.std(), spatial_mask.min(), spatial_mask.max(), spatial_mask.shape)

        # cfg interval
        low, high = model_jit.cfg_interval
        interval_mask = (t < high) & ((low == 0) | (t > low))
        cfg_scale_interval = torch.where(
            interval_mask,
            torch.as_tensor(cfg_scale, device=device, dtype=z.dtype),
            torch.as_tensor(1.0, device=device, dtype=z.dtype),
        )

        # spatially reweighted CFG on v-space
        v_pred = v_uncond + cfg_scale_interval * spatial_mask * (v_cond - v_uncond)

        return v_pred, x_uncond, x_cond, spatial_mask

    def euler_step_with_x0(z, t, t_next, labels):
        v_pred, x_uncond, x_cond, spatial_mask = forward_sample_with_x0(z, t, labels)
        z_next = z + (t_next - t) * v_pred
        return z_next, x_uncond, x_cond, spatial_mask

    def heun_step_with_x0(z, t, t_next, labels):
        v_pred_t, x_uncond_t, x_cond_t, spatial_mask = forward_sample_with_x0(z, t, labels)

        z_next_euler = z + (t_next - t) * v_pred_t
        v_pred_t_next, _, _, _ = forward_sample_with_x0(z_next_euler, t_next, labels)

        v_pred = 0.5 * (v_pred_t + v_pred_t_next)
        z_next = z + (t_next - t) * v_pred
        return z_next, x_uncond_t, x_cond_t, spatial_mask

    if method not in ["euler", "heun"]:
        raise NotImplementedError(f"Unknown sampling method: {method}")

    for i in range(steps - 1):
        t = timesteps[i]
        t_next = timesteps[i + 1]

        if method == "euler":
            z, x0_uncond, x0_cond, spatial_mask = euler_step_with_x0(z, t, t_next, labels)
        else:
            z, x0_uncond, x0_cond, spatial_mask = heun_step_with_x0(z, t, t_next, labels)

        if return_traj:
            traj_x0_uncond.append(x0_uncond.detach().clone())
            traj_x0_cond.append(x0_cond.detach().clone())

        if model_ctm is not None:
            pred_u, cert_u, attn_u = run_ctm_on_jit_batch(x0_uncond)
            pred_c, cert_c, attn_c = run_ctm_on_jit_batch(x0_cond)

            ctm_predictions_uncond.append(pred_u)
            ctm_certainties_uncond.append(cert_u)
            ctm_attention_256_uncond.append(attn_u)

            ctm_predictions_cond.append(pred_c)
            ctm_certainties_cond.append(cert_c)
            ctm_attention_256_cond.append(attn_c)
            ctm_spatial_mask_cond.append(spatial_mask.detach().cpu())

    # last step always euler
    z, x0_uncond, x0_cond, spatial_mask = euler_step_with_x0(
        z, timesteps[-2], timesteps[-1], labels
    )

    if return_traj:
        traj_x0_uncond.append(x0_uncond.detach().clone())
        traj_x0_cond.append(x0_cond.detach().clone())
    else:
        traj_x0_uncond = None
        traj_x0_cond = None

    if model_ctm is not None:
        pred_u, cert_u, attn_u = run_ctm_on_jit_batch(x0_uncond)
        pred_c, cert_c, attn_c = run_ctm_on_jit_batch(x0_cond)

        ctm_predictions_uncond.append(pred_u)
        ctm_certainties_uncond.append(cert_u)
        ctm_attention_256_uncond.append(attn_u)

        ctm_predictions_cond.append(pred_c)
        ctm_certainties_cond.append(cert_c)
        ctm_attention_256_cond.append(attn_c)
        ctm_spatial_mask_cond.append(spatial_mask.detach().cpu())
    else:
        ctm_predictions_uncond = None
        ctm_certainties_uncond = None
        ctm_attention_256_uncond = None

        ctm_predictions_cond = None
        ctm_certainties_cond = None
        ctm_attention_256_cond = None

    ctm_outputs = {
        "uncond": {
            "predictions": ctm_predictions_uncond,
            "certainties": ctm_certainties_uncond,
            "attention_256": ctm_attention_256_uncond,
        },
        "cond": {
            "predictions": ctm_predictions_cond,
            "certainties": ctm_certainties_cond,
            "attention_256": ctm_attention_256_cond,
            "spatial_mask": ctm_spatial_mask_cond,
        },
    }

    return z, traj_x0_uncond, traj_x0_cond, ctm_outputs

In [21]:
# def save_traj_grid_compare

def save_traj_grid_compare(
    traj_uncond,
    traj_cond,
    sampled_images,
    ctm_outputs,
    save_dir,
    prefix="traj",
    nrow=None,
    gap=20,
    title_h=40,
    gif_name=None,
    gif_duration=0.15,
    overlay_alpha=0.45,
):
    """
    Layout (3 columns x 2 rows):
        row 1:
            col 1: uncond tweedie
            col 2: cond tweedie
            col 3: generated images (static)
        row 2:
            col 1: uncond attention overlay
            col 2: cond attention overlay
            col 3: cond spatial mask
    """
    os.makedirs(save_dir, exist_ok=True)

    assert len(traj_uncond) == len(traj_cond)
    assert len(traj_uncond) > 0
    assert len(ctm_outputs["uncond"]["attention_256"]) == len(traj_uncond)
    assert len(ctm_outputs["cond"]["attention_256"]) == len(traj_cond)

    if "spatial_mask" in ctm_outputs["cond"] and ctm_outputs["cond"]["spatial_mask"] is not None:
        assert len(ctm_outputs["cond"]["spatial_mask"]) == len(traj_cond)

    steps = len(traj_uncond)
    B, C, H, W = traj_uncond[0].shape

    if nrow is None:
        nrow = int(math.sqrt(B))
        if nrow * nrow < B:
            nrow += 1
    ncol = math.ceil(B / nrow)

    grid_h = ncol * H
    grid_w = nrow * W

    panel_h = title_h + grid_h
    panel_w = grid_w

    canvas_h = panel_h * 2 + gap
    canvas_w = panel_w * 3 + gap * 2

    def to_rgb_uint8_batch(imgs):
        """
        imgs: tensor [B,3,H,W], in [-1,1]
        return: numpy [B,H,W,3], uint8 RGB
        """
        imgs = (imgs + 1) / 2
        imgs = imgs.clamp(0, 1)
        imgs = imgs.detach().cpu().numpy()
        imgs = np.transpose(imgs, (0, 2, 3, 1))
        imgs = np.round(imgs * 255).astype(np.uint8)
        return imgs

    def make_rgb_grid(imgs):
        """
        imgs: tensor [B,3,H,W], in [-1,1]
        return: uint8 RGB grid [grid_h, grid_w, 3]
        """
        imgs = to_rgb_uint8_batch(imgs)
        grid = np.zeros((grid_h, grid_w, 3), dtype=np.uint8)

        for i in range(B):
            r = i // nrow
            c = i % nrow
            grid[
                r * H:(r + 1) * H,
                c * W:(c + 1) * W
            ] = imgs[i]

        return grid

    def normalize_map_per_image(x):
        x = x.astype(np.float32)
        x_min = float(x.min())
        x_max = float(x.max())
        if x_max > x_min:
            x = (x - x_min) / (x_max - x_min)
        else:
            x = np.zeros_like(x, dtype=np.float32)
        return x

    def make_attention_overlay_grid(imgs, attn_maps):
        """
        imgs: tensor [B,3,H,W], in [-1,1]
        attn_maps: tensor/np.array [B,H,W]
        return: uint8 RGB grid [grid_h, grid_w, 3]
        """
        imgs_rgb = to_rgb_uint8_batch(imgs)

        if isinstance(attn_maps, torch.Tensor):
            attn_maps = attn_maps.detach().cpu().numpy()

        grid = np.zeros((grid_h, grid_w, 3), dtype=np.uint8)

        for i in range(B):
            r = i // nrow
            c = i % nrow

            base_img = imgs_rgb[i]  # RGB uint8
            attn = normalize_map_per_image(attn_maps[i])

            attn_uint8 = np.round(attn * 255).astype(np.uint8)
            heat_bgr = cv2.applyColorMap(attn_uint8, cv2.COLORMAP_JET)
            heat_rgb = cv2.cvtColor(heat_bgr, cv2.COLOR_BGR2RGB)

            overlay = cv2.addWeighted(
                base_img, 1.0 - overlay_alpha,
                heat_rgb, overlay_alpha,
                0
            )

            grid[
                r * H:(r + 1) * H,
                c * W:(c + 1) * W
            ] = overlay

        return grid

    def make_heatmap_grid(single_channel_maps):
        """
        single_channel_maps: tensor/np.array [B,H,W] or [B,1,H,W]
        return: uint8 RGB grid [grid_h, grid_w, 3]
        """
        if isinstance(single_channel_maps, torch.Tensor):
            single_channel_maps = single_channel_maps.detach().cpu().numpy()

        if single_channel_maps.ndim == 4:
            single_channel_maps = single_channel_maps[:, 0]  # [B,H,W]

        grid = np.zeros((grid_h, grid_w, 3), dtype=np.uint8)

        for i in range(B):
            r = i // nrow
            c = i % nrow

            x = normalize_map_per_image(single_channel_maps[i])
            x_uint8 = np.round(x * 255).astype(np.uint8)
            heat_bgr = cv2.applyColorMap(x_uint8, cv2.COLORMAP_JET)
            heat_rgb = cv2.cvtColor(heat_bgr, cv2.COLOR_BGR2RGB)

            grid[
                r * H:(r + 1) * H,
                c * W:(c + 1) * W
            ] = heat_rgb

        return grid

    def draw_title(canvas_bgr, text, x0, y0):
        font = cv2.FONT_HERSHEY_SIMPLEX
        font_scale = 0.7
        thickness = 2

        (tw, th), _ = cv2.getTextSize(text, font, font_scale, thickness)
        x = x0 + max((panel_w - tw) // 2, 5)
        y = y0 + max((title_h + th) // 2, th + 5)

        cv2.putText(
            canvas_bgr,
            text,
            (x, y),
            font,
            font_scale,
            (255, 255, 255),
            thickness,
            cv2.LINE_AA,
        )

    grid_generated = make_rgb_grid(sampled_images)
    saved_paths = []

    for step in range(steps):
        grid_uncond = make_rgb_grid(traj_uncond[step])
        grid_cond = make_rgb_grid(traj_cond[step])

        # attention_256[step]: [T_ctm, B, 256, 256]
        attn_uncond_step = ctm_outputs["uncond"]["attention_256"][step].mean(dim=0)  # [B,256,256]
        attn_cond_step = ctm_outputs["cond"]["attention_256"][step].mean(dim=0)      # [B,256,256]

        grid_attn_uncond = make_attention_overlay_grid(traj_uncond[step], attn_uncond_step)
        grid_attn_cond = make_attention_overlay_grid(traj_cond[step], attn_cond_step)

        # spatial mask: [B,1,H,W]
        if "spatial_mask" in ctm_outputs["cond"] and ctm_outputs["cond"]["spatial_mask"] is not None:
            spatial_mask_step = ctm_outputs["cond"]["spatial_mask"][step]
            grid_spatial_mask = make_heatmap_grid(spatial_mask_step)
            if isinstance(spatial_mask_step, torch.Tensor):
                spatial_mask_mean = float(spatial_mask_step.mean().item())
            else:
                spatial_mask_mean = float(np.mean(spatial_mask_step))
        else:
            grid_spatial_mask = np.zeros((grid_h, grid_w, 3), dtype=np.uint8)
            spatial_mask_mean = 0.0

        cert_uncond_step = ctm_outputs["uncond"]["certainties"][step]
        cert_cond_step = ctm_outputs["cond"]["certainties"][step]

        if isinstance(cert_uncond_step, np.ndarray):
            cert_uncond_step = torch.from_numpy(cert_uncond_step)
        if isinstance(cert_cond_step, np.ndarray):
            cert_cond_step = torch.from_numpy(cert_cond_step)

        cert_uncond_mean = float(cert_uncond_step[:, 1, :].mean().item())
        cert_cond_mean = float(cert_cond_step[:, 1, :].mean().item())

        canvas = np.zeros((canvas_h, canvas_w, 3), dtype=np.uint8)

        # row 1
        y0, x0 = 0, 0
        canvas[y0 + title_h:y0 + title_h + grid_h, x0:x0 + grid_w] = grid_uncond

        y0, x0 = 0, panel_w + gap
        canvas[y0 + title_h:y0 + title_h + grid_h, x0:x0 + grid_w] = grid_cond

        y0, x0 = 0, (panel_w + gap) * 2
        canvas[y0 + title_h:y0 + title_h + grid_h, x0:x0 + grid_w] = grid_generated

        # row 2
        y0, x0 = panel_h + gap, 0
        canvas[y0 + title_h:y0 + title_h + grid_h, x0:x0 + grid_w] = grid_attn_uncond

        y0, x0 = panel_h + gap, panel_w + gap
        canvas[y0 + title_h:y0 + title_h + grid_h, x0:x0 + grid_w] = grid_attn_cond

        y0, x0 = panel_h + gap, (panel_w + gap) * 2
        canvas[y0 + title_h:y0 + title_h + grid_h, x0:x0 + grid_w] = grid_spatial_mask

        canvas_bgr = canvas[:, :, ::-1].copy()

        draw_title(canvas_bgr, f"uncond tweedie | jit timestep {step:03d}", 0, 0)
        draw_title(canvas_bgr, f"cond tweedie | jit timestep {step:03d}", panel_w + gap, 0)
        draw_title(canvas_bgr, "generated images", (panel_w + gap) * 2, 0)

        draw_title(
            canvas_bgr,
            f"uncond attention overlay | jit timestep {step:03d} | cert2 avg {cert_uncond_mean:.4f}",
            0,
            panel_h + gap,
        )
        draw_title(
            canvas_bgr,
            f"cond attention overlay | jit timestep {step:03d} | cert2 avg {cert_cond_mean:.4f}",
            panel_w + gap,
            panel_h + gap,
        )
        draw_title(
            canvas_bgr,
            f"cond spatial mask | jit timestep {step:03d} | mean {spatial_mask_mean:.4f}",
            (panel_w + gap) * 2,
            panel_h + gap,
        )

        filename = f"{prefix}_step{step:03d}.png"
        path = os.path.join(save_dir, filename)
        cv2.imwrite(path, canvas_bgr)
        saved_paths.append(path)

    print('fig saved')

    if gif_name is None:
        gif_name = f"{prefix}.gif"

    gif_path = os.path.join(save_dir, gif_name)

    frames = []
    for p in saved_paths:
        frame_bgr = cv2.imread(p, cv2.IMREAD_COLOR)
        frame_rgb = cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB)
        frames.append(frame_rgb)

    imageio.mimsave(gif_path, frames, duration=gif_duration, loop=0)
    print('gif saved')

    return gif_path

In [24]:
with torch.inference_mode():
    set_seed(124)
    labels = torch.arange(0, 20, dtype=torch.int64, device=device)
    # labels = torch.tensor([12,13,14,15,16,17,18,19,20,21,22,23,24,25], dtype=torch.int64, device=device)
    
    sampled_images, traj_x0_uncond, traj_x0_cond, ctm_outputs = sample_ctm_jit(
        model_jit = model_jit,
        labels= labels,
        sample_steps = 50,
        cfg_scale = 3,
        method = "heun",
        return_traj = True,
        model_ctm = model_ctm,
        ctm_input_size = 224,
        attn_output_size = 256,
        attn_mask_strength = 0.25,
        attn_mask_clip_min = 0.75,
        attn_mask_clip_max = 1.25,
    )




In [25]:
gif_path = save_traj_grid_compare(
    traj_uncond=traj_x0_uncond,
    traj_cond=traj_x0_cond,
    sampled_images=sampled_images,
    ctm_outputs=ctm_outputs,
    save_dir="tweedie_trajs",
    prefix="jit_ctm_compare",
    gif_duration=2.,
)
print(gif_path)

fig saved
gif saved
tweedie_trajs/jit_ctm_compare.gif
